In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# ============================================================
# BLOCK 1 : LOAD MODEL & DATASET
# ============================================================

!pip -q install -U ultralytics

import os
import cv2
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from ultralytics import YOLO

# ============================================================
# PATHS
# ============================================================

WEIGHTS = "/kaggle/input/datasets/garvitpujari/weight-file/best (2).pt"

VIDEO_ROOT = "/kaggle/input/datasets/bninaayoub/dut-anti-uav-tracking-dataset/Anti-UAV-Tracking-V0/Anti-UAV-Tracking-V0"

OUTPUT_DIR = "/kaggle/working"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# LOAD MODEL
# ============================================================

print("="*70)
print("Loading Production YOLOv11 Model")
print("="*70)

model = YOLO(WEIGHTS)

print("✓ Weights Loaded Successfully")

# ============================================================
# GPU
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("\nDevice :", DEVICE)

if DEVICE == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

# ============================================================
# LOAD VIDEO LIST
# ============================================================

VIDEOS = sorted([
    d for d in os.listdir(VIDEO_ROOT)
    if os.path.isdir(os.path.join(VIDEO_ROOT, d))
])

print("\nTotal Videos :", len(VIDEOS))

assert len(VIDEOS) == 20, "Expected 20 DUT videos."

print("\nVideos")

for v in VIDEOS:
    print(v)

# ============================================================
# CSV STORAGE
# ============================================================

records = []

print("\nReady for Feature Extraction.")

Loading Production YOLOv11 Model
✓ Weights Loaded Successfully

Device : cuda
GPU : Tesla T4

Total Videos : 20

Videos
video01
video02
video03
video04
video05
video06
video07
video08
video09
video10
video11
video12
video13
video14
video15
video16
video17
video18
video19
video20

Ready for Feature Extraction.


In [3]:
# ============================================================
# BLOCK 2 : EXTRACT TRACKING DATA
# ============================================================

from pathlib import Path

print("="*70)
print("Generating Tracking Dataset")
print("="*70)

for video_name in VIDEOS:

    print(f"\nProcessing {video_name}")

    image_folder = os.path.join(VIDEO_ROOT, video_name)

    images = sorted([
        f for f in os.listdir(image_folder)
        if f.endswith(".jpg")
    ])

    for frame_no, img_name in enumerate(tqdm(images), start=1):

        img_path = os.path.join(image_folder, img_name)

        results = model.track(

            source=img_path,

            tracker="bytetrack.yaml",

            persist=True,

            verbose=False,

            conf=0.25,

            iou=0.5
        )

        result = results[0]

        if result.boxes is None:
            continue

        boxes = result.boxes

        for box in boxes:

            # Skip if ByteTrack hasn't assigned an ID yet
            if box.id is None:
                continue

            track_id = int(box.id.item())

            conf = float(box.conf.item())

            x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]

            w = x2 - x1
            h = y2 - y1

            cx = x1 + w / 2
            cy = y1 + h / 2

            records.append({

                "video_id": video_name,

                "frame_id": frame_no,

                "track_id": track_id,

                "confidence": conf,

                "x1": float(x1),
                "y1": float(y1),
                "x2": float(x2),
                "y2": float(y2),

                "cx": float(cx),
                "cy": float(cy),

                "width": float(w),
                "height": float(h)

            })

print("\n" + "="*70)
print("Tracking Extraction Finished")
print("="*70)

print("Total Records :", len(records))

Generating Tracking Dataset

Processing video01


  0%|          | 0/1050 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 286ms
Prepared 1 package in 67ms
Installed 1 package in 6ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|██████████| 1050/1050 [00:44<00:00, 23.61it/s]



Processing video02


...


Processing video19


100%|██████████| 1300/1300 [00:45<00:00, 28.68it/s]



Processing video20


100%|██████████| 1635/1635 [00:57<00:00, 28.51it/s]


Tracking Extraction Finished
Total Records : 21085


In [2]:
# ============================================================
# BLOCK 3 : CREATE MOTION FEATURES
# ============================================================

import numpy as np
import pandas as pd

print("="*70)
print("Creating Motion Features")
print("="*70)

df = pd.DataFrame(records)

# ----------------------------------------------------------
# Sort
# ----------------------------------------------------------

df = df.sort_values(
    ["video_id","track_id","frame_id"]
).reset_index(drop=True)

# ----------------------------------------------------------
# Velocity
# ----------------------------------------------------------

df["vx"] = df.groupby(
    ["video_id","track_id"]
)["cx"].diff()

df["vy"] = df.groupby(
    ["video_id","track_id"]
)["cy"].diff()

# ----------------------------------------------------------
# Speed
# ----------------------------------------------------------

df["speed"] = np.sqrt(
    df["vx"]**2 +
    df["vy"]**2
)

# ----------------------------------------------------------
# Heading
# ----------------------------------------------------------

df["heading"] = np.arctan2(
    df["vy"],
    df["vx"]
)

# ----------------------------------------------------------
# Acceleration
# ----------------------------------------------------------

df["acceleration"] = df.groupby(
    ["video_id","track_id"]
)["speed"].diff()

# ----------------------------------------------------------
# Remove first frame of every trajectory
# ----------------------------------------------------------

df = df.dropna().reset_index(drop=True)

# ----------------------------------------------------------
# Save CSV
# ----------------------------------------------------------

csv_path = "/kaggle/working/tracking_dataset.csv"

df.to_csv(csv_path,index=False)

print("\nTracking Dataset Saved Successfully")

print("\nLocation")
print(csv_path)

print("\nDataset Shape :",df.shape)

print("\nColumns\n")
print(df.columns.tolist())

print("\nFirst Five Rows\n")

display(df.head())

Creating Motion Features


NameError: name 'records' is not defined

In [1]:
# ============================================================
# VERIFY MOTION CALCULATION FOR ONE TRACK
# ============================================================

track = (
    df[
        (df["video_id"] == "video01") &
        (df["track_id"] == 1)
    ]
    .copy()
    .reset_index(drop=True)
)

display(track.head(10)[[
    "frame_id",
    "cx",
    "cy",
    "vx",
    "vy",
    "speed",
    "acceleration"
]])

NameError: name 'df' is not defined

In [4]:
# ============================================================
# LOAD TRACKING DATASET
# ============================================================

import pandas as pd

CSV_PATH = "/kaggle/input/datasets/garvitpujari/trackingdataset/tracking_dataset.csv"

df = pd.read_csv(CSV_PATH)

print("="*60)
print("Tracking Dataset Loaded")
print("="*60)

print("Shape :", df.shape)

print("\nColumns\n")
print(df.columns.tolist())

display(df.head())

Tracking Dataset Loaded
Shape : (20799, 17)

Columns

['video_id', 'frame_id', 'track_id', 'confidence', 'x1', 'y1', 'x2', 'y2', 'cx', 'cy', 'width', 'height', 'vx', 'vy', 'speed', 'heading', 'acceleration']


,video_id,frame_id,track_id,confidence,x1,y1,x2,y2,cx,cy,width,height,vx,vy,speed,heading,acceleration
0,video01,3,1,0.904163,928.097107,559.881409,1173.236816,702.410034,1050.666992,631.145752,245.139709,142.528625,-1.981934,0.039978,1.982337,3.121424,1.782492
1,video01,4,1,0.896066,925.846802,559.630310,1169.954468,701.614258,1047.900635,630.622314,244.107666,141.983948,-2.766357,-0.523438,2.815443,-2.954588,0.833106
2,video01,5,1,0.893714,922.317932,560.483398,1164.443481,701.389648,1043.380737,630.936523,242.125549,140.906250,-4.519897,0.314209,4.530806,3.072187,1.715363
3,video01,6,1,0.891884,919.882690,561.700684,1157.249023,699.817017,1038.565918,630.758850,237.366333,138.116333,-4.814819,-0.177673,4.818096,-3.104708,0.287291
4,video01,7,1,0.892384,916.480042,562.464844,1151.115845,699.043884,1033.797974,630.754395,234.635803,136.579041,-4.767944,-0.004456,4.767946,-3.140658,-0.050150


In [5]:
# ============================================================
# VERIFY VELOCITY & ACCELERATION MANUALLY
# ============================================================

track = (
    df[
        (df["video_id"] == "video01") &
        (df["track_id"] == 1)
    ]
    .copy()
    .reset_index(drop=True)
)

# Manual computation
track["manual_vx"] = track["cx"].diff()
track["manual_vy"] = track["cy"].diff()

track["manual_speed"] = (
    track["manual_vx"]**2 +
    track["manual_vy"]**2
) ** 0.5

track["manual_acceleration"] = track["manual_speed"].diff()

display(
    track[
        [
            "frame_id",
            "cx",
            "cy",

            "vx",
            "manual_vx",

            "vy",
            "manual_vy",

            "speed",
            "manual_speed",

            "acceleration",
            "manual_acceleration"
        ]
    ].head(10)
)

,frame_id,cx,cy,vx,manual_vx,vy,manual_vy,speed,manual_speed,acceleration,manual_acceleration
0,3,1050.666992,631.145752,-1.981934,NaN,0.039978,NaN,1.982337,NaN,1.782492,NaN
1,4,1047.900635,630.622314,-2.766357,-2.766357,-0.523438,-0.523438,2.815443,2.815443,0.833106,NaN
2,5,1043.380737,630.936523,-4.519897,-4.519897,0.314209,0.314209,4.530806,4.530806,1.715363,1.715363
3,6,1038.565918,630.758850,-4.814819,-4.814819,-0.177673,-0.177673,4.818096,4.818096,0.287291,0.287291
4,7,1033.797974,630.754395,-4.767944,-4.767944,-0.004456,-0.004456,4.767946,4.767946,-0.050150,-0.050150
5,8,1032.279053,630.760010,-1.518921,-1.518921,0.005615,0.005615,1.518931,1.518931,-3.249015,-3.249015
6,9,1028.815918,628.898071,-3.463135,-3.463135,-1.861938,-1.861938,3.931936,3.931936,2.413004,2.413004
7,10,1023.374634,628.246826,-5.441284,-5.441284,-0.651245,-0.651245,5.480118,5.480118,1.548182,1.548182
8,11,1017.290222,626.650879,-6.084412,-6.084412,-1.595947,-1.595947,6.290239,6.290239,0.810121,0.810121
9,12,1012.550659,624.179993,-4.739563,-4.739563,-2.470886,-2.470886,5.344973,5.344973,-0.945266,-0.945266
